# 来歴情報を編集する
来歴情報を編集するタスクです。<br>
このサブフロー内に保存されているデータの来歴情報を編集します。<br>
データの追加や削除を行った場合や、来歴情報の編集を行いたい場合に実行してください。

## ディレクトリを表示する

In [ ]:
# ディレクトリを表示する
import os

import panel as pn
from IPython.core.display import Javascript
from IPython.display import display

from library.utils.access import open_data_folder

button = open_data_folder(os.path.abspath('__file__'))

pn.extension()
display(button)
display(Javascript('IPython.notebook.save_checkpoint();'))

## 来歴情報を編集するフォームを表示する
来歴情報を編集するためのフォームを表示します。<br>
<span style="color:red">※　次のセルを実行するとGRDMとの同期が行われます。</span>


In [ ]:
# 来歴情報を編集する
import os
import traceback

from library.utils.storage_provider import grdm
import panel as pn
from requests.exceptions import RequestException

from library.task_director import TaskDirector
from library.utils.access import create_single_file_selector
from library.utils.config import connect as con_config
from library.utils.config import path_config, message as msg_config
from library.utils.error import (NotFoundSubflowDataError, UnusableVault, ProjectNotExist,
                                    UnauthorizedError, RepoPermissionError)
from library.utils.widgets import Button, MessageBox
from library.utils.setting import get_data_dir
from library.utils.input import get_grdm_connection_parameters
from IPython.core.display import Javascript
from library.utils.research_flow_provenance.prov import ProvenanceManager

notebook_name = 'organize_paper_and_argument_data.ipynb.ipynb'

class EditProvenanceData(TaskDirector):
    """親となる実験サブフローと実行中のサブフローの論拠データのフォルダを表示するクラスです。

    Attributes:
        instance:
            working_path(str): 実行Notebookファイルパス
            _msg_output(MessageBox): メッセージ出力用のボックス
            _form_section(pn.WidgetBox): ボタン等の出力を格納するためのボックス

    """
    def __init__(self, working_path: str):
        """Displyクラスのコンストラクタです。

        Args:
            working_path (str): 実行Notebookファイルパス

        """
        self.working_path = working_path
        super().__init__(self.working_path, notebook_name)

        self.grdm_url = con_config.get('GRDM', 'BASE_URL')
        self.grdm = grdm.Grdm()

        pn.extension()

        # 出力用フォームの設定
        self._form_section = pn.WidgetBox()
        # 実行結果出力用メッセージボックスの設定
        self._msg_output = MessageBox()
        self._msg_output.width = 900

        self.warning_area = pn.Row()
        self.warning_area.styles = {'background': '#ffe5e5'}

        display(self._form_section)

    def get_grdm_params(self) -> tuple[str, str]:
        """GRDMのトークンとプロジェクトIDを取得するメソッドです。

        Returns:
            str:GRDMのトークンの値を返す。
            str:プロジェクトIDの値を返す。
        """
        token = ""
        project_id = ""
        try:
            token, project_id = get_grdm_connection_parameters(self.grdm_url)
        except UnusableVault as e:
            message = msg_config.get('form', 'no_vault')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except RepoPermissionError:
            message = msg_config.get('form', 'insufficient_permission')
            self._msg_output.update_error(message)
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except ProjectNotExist as e:
            self._msg_output.update_error(str(e))
            self.log.error(traceback.format_exc())
        except RequestException as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
        return token, project_id

    async def sync_grdm(self):
        """GRDMにサブフローデータを同期する関数です。"""

        try:
            data_dir = os.path.join(get_data_dir(self.working_path), "argument_data")
            abs_path =  os.path.abspath(data_dir)
            await self.grdm.sync(
                    token=self.token,
                    base_url=self.grdm_url,
                    project_id=self.project_id,
                    abs_source=abs_path,
                    abs_root=self._abs_root_path
                )
        except UnauthorizedError:
            message = msg_config.get('form', 'token_unauthorized')
            self._msg_output.update_warning(message)
            self.log.warning(f'{message}\n{traceback.format_exc()}')
            return
        except RequestException as e:
            message = msg_config.get('DEFAULT', 'connection_error')
            self._msg_output.update_error(f'{message}\n{str(e)}')
            self.log.error(f'{message}\n{traceback.format_exc()}')
            return
        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)
            return

    @TaskDirector.task_cell("9")
    async def generate_file_selector(self):
        """データフォルダを表示するためのボタンを生成するメソッドです。"""

        self.doing_task()
        self._form_section.append(self._msg_output)
        #サブフローのデータをGRDMに同期する
        try:
            self._msg_output.update_info("同期中")
            self.token, self.project_id = self.get_grdm_params()
            await self.sync_grdm()
            self._msg_output.update_success("同期完了")

            data_dir = os.path.join(get_data_dir(self.working_path), "argument_data")
            abs_path =  os.path.abspath(data_dir)

            self.prov_manager = ProvenanceManager(self.token, self.grdm_url, self.project_id)

            # ファイルが存在するかの検証を行う
            error_files = self.prov_manager.check_file_exist(abs_path)

            if error_files:
                alerts = [self.create_file_alert(path, ids) for path, ids in error_files.items()]
                alart_panel = pn.Column(*alerts, max_height=300, scroll=True)

            # 全ファイルを選択可能な状態で表示する。
            self.selector, self.relative_path, self.checkbox_dict = create_single_file_selector(abs_path)

            self.edit_button = Button(width=500)
            self.edit_button.set_looks_init("編集する")
            self.edit_button.on_click(self. _handle_click)

        except Exception:
            message = f'## [INTERNAL ERROR] : {traceback.format_exc()}'
            self._msg_output.update_error(message)
            self.log.error(message)

        self.done_task()
        # 表示する
        self.log.error("ここまで１")
        self._form_section.append(self.selector)
        self._form_section.append(self.edit_button)
        display(Javascript('IPython.notebook.save_checkpoint();'))


    async def _handle_click(self, event):
        """非同期処理を実行するための仲介メソッドです。"""
        await self._edit(event)

    async def _edit(self, event):
        """編集を実行する"""
        self.log.error("run")
        clear_output()
        selected_file = None
        for file_path, checkbox in self.checkbox_dict.items():
            if checkbox.value:
                selected_file = file_path
                break
        edit_title = pn.pane.Markdown(f"###{selected_file}の来歴情報を編集する", width=500)

        edit_option = {
            "コピー": "File Copy",
            "アップロード" : "File Upload"
        }
        self.edit_selector = pn.widgets.Select(name="関連付ける関係性を選択してください", options=edit_option)
        self.edit_selector.param.watch(_handle_selected, 'value')

        self._form_section.append(edit_title)
        self._form_section.append(self.edit_selector)


        def _handle_selected():
            if self.edit_selector.value == "File Copy":
                generate_copy_widget()

            elif self.edit_selector.value == "File Upload":
                generate_upload_widget()

            def generate_copy_widget(self):
                """コピー用のウィジェットを生成する"""

            def generate_upload_widget(self):
                """アップロード用のウィジェットを生成する"""

    def create_file_alert(self, file_path, ids):
        alert = pn.widgets.Alert(
            f"ファイルが存在しません: {file_path}",
            alert_type="danger",
            closable=False,
        )
        delete_button = pn.widgets.Button(name="削除", button_type="danger", width=80)
        change_button = pn.widgets.Button(name="パス変更", button_type="primary", width=80)

        # 新しいパス入力用ウィジェット
        new_path_input = pn.widgets.TextInput(name='新しいパスを入力してください', visible=False, width=300)
        confirm_change_button = pn.widgets.Button(name="変更確定", button_type="success", visible=False, width=80)

        # ボタン押下時の処理
        def on_delete(event):
            for id in ids:
                self.prov_manager.handle("File Delete", id)
            alert.object = f"{file_path} を削除しました"
            alert.alert_type = "success"
            delete_button.visible = False
            change_button.visible = False

        def on_change(event):
            new_path_input.visible = True
            confirm_change_button.visible = True
            change_button.disabled = True  # 重複押し防止

        def on_confirm_change(event):
            new_path = new_path_input.value.strip()
            if new_path:
                # パスが存在するかを調べた後、関数を実行
                alert.object = f"{file_path} のパスを {new_path} に変更しました"
                alert.alert_type = "success"
                # ここに実際のパス変更処理を書く
                # ボタンや入力欄を隠す
                new_path_input.visible = False
                confirm_change_button.visible = False
                delete_button.visible = False
                change_button.visible = False
            else:
                alert.object = "新しいパスを入力してください"
                alert.alert_type = "warning"

        delete_button.on_click(on_delete)
        change_button.on_click(on_change)
        confirm_change_button.on_click(on_confirm_change)

        # アラートとボタンを横並びに表示
        return pn.Column(
        pn.Row(alert, delete_button, change_button),
        pn.Row(new_path_input, confirm_change_button)
        )

await EditProvenanceData(os.path.abspath('__file__')).generate_file_selector()


## GakuNin RDMに保存する

In [ ]:
# GakuNin RDMに保存する
import os
from IPython.core.display import Javascript
from IPython.display import display

import panel as pn

from library.utils.config import path_config
from library.task_director import TaskDirector
from library.utils.setting import get_data_dir

script_file_name = 'organize_paper_and_argument_data'
notebook_name = script_file_name+'.ipynb'


class DataSaver(TaskDirector):
    """GRDMに保存するクラスです。

    Attributes:
        instance:
            _abs_root_path (str): 絶対パス
            save_form_box(pn.WidgetBox):フォームを格納する。
            save_msg_output(Message):ユーザーに提示するメッセージを格納する。

    """

    def __init__(self, working_path: str) -> None:
        """DataSaver コンストラクタメソッドです。

        Args:
            working_path (str): 実行Notebookファイルパス
        """
        self.working_path = working_path
        super().__init__(self.working_path, notebook_name)

    @TaskDirector.task_cell("10")
    def generate_form_section(self):
        """取得したデータを表示するメソッドです。"""
        # タスク開始によるサブフローステータス管理JSONの更新

        # フォーム定義
        data_dir = get_data_dir(self.working_path)
        source = [data_dir]
        if os.path.exists(os.path.join(data_dir, 'README.md')):
            source.append(os.path.join(data_dir, 'README.md'))
        self.define_save_form(source)
        # フォーム表示
        pn.extension()
        form_section = pn.WidgetBox()
        form_section.append(self.save_form_box)
        form_section.append(self.save_msg_output)
        display(form_section)
        display(Javascript('IPython.notebook.save_checkpoint();'))


DataSaver(working_path=os.path.abspath('__file__')).generate_form_section()

## サブフローメニューを表示する

In [ ]:
# サブフローメニューを表示する
import os

from library.task_director import TaskDirector

script_file_name = "organize_paper_and_argument_data"
notebook_name = script_file_name+'.ipynb'

TaskDirector(os.path.abspath('__file__'), notebook_name).return_subflow_menu()